# 02 - SFT (skill 6)
LoRA SFT of Tiny Aya on the oracle's gold trajectories (assistant-only loss), with validation on a
fixed 90-scenario val subset at every checkpoint and a safety-constrained checkpoint choice.
Needs the baseline results (skill 5) in `results/metrics/` for the base unsafe rate; pull them from
the HF dataset repo first. The run is resumable from the last checkpoint (`runs/sft`).

In [ ]:
# 1. GPU + parameters
!nvidia-smi
import os, sys
REPO_URL = "https://github.com/YOUR_GITHUB_USER/tinyops"
HF_USER = "YOUR_HF_USERNAME"
DATA_REPO = f"{HF_USER}/tinyops-data"
SFT_REPO = f"{HF_USER}/tinyops-tiny-aya-sft"
PER_DEVICE_BATCH = 1      # T4: 1, L4/A10G: 2
INJECTED_OVERSAMPLE = 1.0 # 2.0 if SFT val unsafe_rate > base (retrain once)

In [ ]:
# 2. install, secrets, environment record
if not os.path.exists("tinyops"):
    !git clone {REPO_URL} tinyops
%cd tinyops
if sys.version_info >= (3, 12):
    PY = "python"
    !pip install -q -e ".[train]"
else:
    PY = "/content/venv/bin/python"
    !pip -q install uv
    !uv venv --python 3.12 /content/venv
    !uv pip install --python {PY} -q -e ".[train]"
from google.colab import userdata
os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
!{PY} -m tinyops.utils.envinfo

In [ ]:
# 3. data: verify locked test hashes, fetch baseline results, regenerate gold (deterministic)
!{PY} -m tinyops.scenarios.build_dataset verify
from huggingface_hub import snapshot_download
snapshot_download(DATA_REPO, repo_type="dataset", allow_patterns=["baselines/metrics/*", "baselines/runs/*"], local_dir="hf_data")
!mkdir -p results/metrics results/runs && cp -n hf_data/baselines/metrics/* results/metrics/ && cp -n hf_data/baselines/runs/* results/runs/
!{PY} -m tinyops.scenarios.build_dataset gold --split all --seed 1234

In [ ]:
# 4. train (stats of the dataset, including how many examples exceed max_seq_len, are logged first)
!{PY} -m tinyops.train.sft --config configs/sft.yaml --set per_device_batch={PER_DEVICE_BATCH} \
    --set push_to_hub={SFT_REPO} --set mix.injected_oversample={INJECTED_OVERSAMPLE}

In [ ]:
# 5. selection rationale and the validation curve
import json, pandas as pd
print(json.dumps(json.load(open("results/runs/sft_selection.json"))["chosen"], indent=2))
pd.read_csv("results/train_logs/sft_val_curve.csv")

In [ ]:
# 6. full val (180) with the selected adapter -> results/metrics/sft_val.json
chosen = json.load(open("results/runs/sft_selection.json"))["chosen"]["checkpoint"]
!{PY} -m tinyops.eval.run_eval --policy hf --model CohereLabs/tiny-aya-global --adapter {chosen} \
    --split val --batch-size 8 --out results/episodes/sft_val.jsonl
print(open("results/metrics/sft_val.md").read())

In [ ]:
# 7. model card draft + push adapter, provenance and logs
!{PY} -m tinyops.train.model_card --repo-id {SFT_REPO}
from huggingface_hub import HfApi
api = HfApi()
api.create_repo(SFT_REPO, exist_ok=True)
api.upload_folder(folder_path=chosen, repo_id=SFT_REPO, ignore_patterns=["optimizer.pt", "*.pth", "rng_state*"])
api.upload_file(path_or_fileobj="hub/model_card_sft.md", path_in_repo="README.md", repo_id=SFT_REPO)
api.create_repo(DATA_REPO, repo_type="dataset", exist_ok=True)
for f in ["results/runs/sft_selection.json", "results/runs/sft_train_run.json", "results/train_logs/sft_val_curve.csv",
          "results/metrics/sft_val.json", "results/compute_log.csv"]:
    api.upload_file(path_or_fileobj=f, path_in_repo=f"training_logs/sft/{os.path.basename(f)}", repo_id=DATA_REPO, repo_type="dataset")

## Decision rules (from the plan)
- SFT val `unsafe_rate` above the base model's: re-run cell 4 with `INJECTED_OVERSAMPLE = 2.0` (once).
- SFT val success below the few-shot baseline: check loss masking and the template first
  (`python -m tinyops.train.sft_data` helpers `decode_trained`), then try `lr=5e-5`.
- Format validity below 95%: look at parse errors (`tinyops.eval.baseline_findings`); usually a
  template/format mismatch between training data and inference.
- If SFT is not working by the SFT deadline, ship the best checkpoint and move on to the DPO path.